# Use Case - Billing.ipynb

## Overview

This notebook demonstrates how MSPs can use metering data from the Sovereign Core platform
to model and communicate billing to their tenants. It uses **sample** `api_calls` usage data
to illustrate four common pricing models used in cloud and managed service billing.

> **Note:** This is an example notebook intended to demonstrate what is possible with Sovereign Core metering data.
> The current demonstration runs on sample data. You can customise the pricing models, rates, and sections to suit your own needs,
> and evolve the notebook to include additional billing strategies — there are many more things you could do beyond what is shown here.
> Rates used here are illustrative demo values — not production pricing.

The goal is not to produce a production billing system — it is to show how the Sovereign Core
metrics-aggregator API provides the usage foundation that any billing model can be built on.

If you need to understand **operational performance** rather than cost, use
**`Use Case - Telemetry.ipynb`** instead.

---

### What this notebook produces

| # | Section | Chart | What it shows |
|---|---|---|---|
| 4.1 | Usage Baseline | Horizontal bar chart | Total `api_calls` per tenant — the billing input |
| 4.2 | Model 1 — PayGo | Vertical bar chart | Cost per tenant at a fixed rate per call |
| 4.3 | Model 2 — Per-instance | Vertical bar chart | Cost per tenant based on average running instances |
| 4.4 | Model 3 — Contract | Gauge indicators | % of committed call volume consumed per tenant |
| 4.5 | Model 4 — Contract + Overage | Stacked bar chart | In-contract vs overage cost split per tenant |
| 4.6 | Model Comparison | Grouped bar chart | All four model costs side-by-side per tenant |

---

### Prerequisites

- This notebook runs out of the box using the **included sample data** — no deployment or API access needed.
- To use your own data, run **`Fetch - Usage Data.ipynb`** first to populate `data/grouped/`, or point the configuration (Section 2) at an existing data directory.

All charts are fully interactive — hover for exact values, drag to zoom, double-click to reset.
Adjust any pricing rate in Section 2 (Configuration) and re-run to explore different scenarios.

> Rates used here are illustrative demo values — not production pricing.


## 1. Imports

In [ ]:
import json
import math
import os
import pathlib

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv
import ipywidgets as widgets
from IPython.display import display, clear_output

## 2. Configuration

Pre-filled with the demo environment. Edit any field and click **Apply Configuration** to update.

Pricing rates are set below the service fields — adjust them to model different scenarios.

| Default | Value |
|---|---|
| `APP_DOMAIN` | `apps.cluster.url.com` |
| `SERVICE_ID` | `servicebrokercore` |

No secrets are loaded or printed here.

In [2]:
DEMO_APP_DOMAIN = "apps.cluster.url.com"
DEMO_SERVICE_ID = "servicebrokercore"

if pathlib.Path(".env").exists():
    load_dotenv(".env", override=False)
else:
    load_dotenv(".env.template", override=False)

# ── Widgets ───────────────────────────────────────────────────────────────────
_style  = {"description_width": "150px"}
_layout = widgets.Layout(width="500px")
_btn_layout = widgets.Layout(width="220px", height="36px", margin="12px 0 0 154px")

w_domain   = widgets.Text(value=DEMO_APP_DOMAIN, description="APP_DOMAIN:",  style=_style, layout=_layout)
w_service  = widgets.Text(value=DEMO_SERVICE_ID, description="SERVICE_ID:",  style=_style, layout=_layout)

w_paygo_rate      = widgets.FloatText(value=0.50,  description="PayGo ($/1M calls):",    style=_style, layout=_layout)
w_instance_rate   = widgets.FloatText(value=200.0, description="Instance ($/period):",   style=_style, layout=_layout)
w_contract_size   = widgets.FloatText(value=20.0,  description="Contract size (M calls):",style=_style, layout=_layout)
w_contract_rate   = widgets.FloatText(value=8.0,   description="Contract ($/1M calls):", style=_style, layout=_layout)
w_overage_rate    = widgets.FloatText(value=0.70,  description="Overage ($/1M calls):",  style=_style, layout=_layout)

w_btn = widgets.Button(description="Apply Configuration", button_style="primary", icon="check", layout=_btn_layout)
w_out = widgets.Output()

# ── Global state ──────────────────────────────────────────────────────────────
app_domain = DEMO_APP_DOMAIN
service_id = DEMO_SERVICE_ID
GRP_DIR    = pathlib.Path(f"data/grouped/{app_domain}/{service_id}")
PAYGO_RATE_PER_CALL    = w_paygo_rate.value    / 1_000_000
INSTANCE_RATE          = w_instance_rate.value
CONTRACT_CALLS         = int(w_contract_size.value * 1_000_000)
CONTRACT_RATE_PER_CALL = w_contract_rate.value  / 1_000_000
OVERAGE_RATE_PER_CALL  = w_overage_rate.value   / 1_000_000

def _apply(_):
    global app_domain, service_id, GRP_DIR
    global PAYGO_RATE_PER_CALL, INSTANCE_RATE, CONTRACT_CALLS, CONTRACT_RATE_PER_CALL, OVERAGE_RATE_PER_CALL
    app_domain = w_domain.value.strip()  or DEMO_APP_DOMAIN
    service_id = w_service.value.strip() or DEMO_SERVICE_ID
    GRP_DIR    = pathlib.Path(os.getenv("GROUPED_DATA_PATH") or f"data/grouped/{app_domain}/{service_id}")
    PAYGO_RATE_PER_CALL    = w_paygo_rate.value    / 1_000_000
    INSTANCE_RATE          = w_instance_rate.value
    CONTRACT_CALLS         = int(w_contract_size.value * 1_000_000)
    CONTRACT_RATE_PER_CALL = w_contract_rate.value  / 1_000_000
    OVERAGE_RATE_PER_CALL  = w_overage_rate.value   / 1_000_000
    with w_out:
        clear_output(wait=True)
        tag = lambda v, d: "  (demo default)" if v == d else "  (custom)"
        print(f"✓ APP_DOMAIN  : {app_domain}{tag(app_domain, DEMO_APP_DOMAIN)}")
        print(f"✓ SERVICE_ID  : {service_id}{tag(service_id, DEMO_SERVICE_ID)}")
        print(f"✓ Grouped data: {GRP_DIR}  ({'found' if list(GRP_DIR.glob('*.json')) else 'NOT FOUND'})")
        print(f"\n── Pricing rates ──")
        print(f"  PayGo             : ${w_paygo_rate.value:.2f} per 1M calls")
        print(f"  Per-instance flat : ${w_instance_rate.value:.2f} per instance per period")
        print(f"  Contract size     : {w_contract_size.value:.1f}M calls")
        print(f"  Contract rate     : ${w_contract_rate.value:.2f} per 1M calls")
        print(f"  Overage rate      : ${w_overage_rate.value:.2f} per 1M calls")

w_btn.on_click(_apply)
display(widgets.VBox([
    widgets.HTML("<b style='font-size:13px'>Service</b>"),
    w_domain, w_service,
    widgets.HTML("<b style='font-size:13px; margin-top:10px'>Pricing rates</b>"),
    w_paygo_rate, w_instance_rate, w_contract_size, w_contract_rate, w_overage_rate,
    w_btn, w_out,
]))
_apply(None)  # auto-apply on first run

## 3. Load Data

Loads the two grouped datasets needed for billing:

| Variable | File pattern | Used for |
|---|---|---|
| `api_df` | `*sum_api_calls_by_tenant*` | All 4 pricing models |
| `inst_df` | `*avg_instances_by_tenant*` | Model 2 (per-instance) |

Both datasets are tenant-grouped daily aggregates from `data/grouped/`.
Run `Fetch - Usage Data.ipynb` first if either shows `⚠ not found`.

In [3]:
def _load_grouped(directory, pattern):
    """Load first file matching pattern; return (DataFrame, params) or (None, None)."""
    matches = sorted(directory.glob(pattern))
    if not matches:
        return None, None
    with open(matches[0]) as f:
        data = json.load(f)
    periods = data.get("aggregatedMeteredUsagePeriods", [])
    if not periods:
        return None, data.get("params", {})
    df = pd.DataFrame(periods)
    df["periodStart"]    = pd.to_datetime(df["periodStart"], unit="ms", utc=True)
    df["periodEnd"]      = pd.to_datetime(df["periodEnd"],   unit="ms", utc=True)
    df["periodQuantity"] = pd.to_numeric(df["periodQuantity"])
    if "groupTenantId" in df.columns:
        df["groupTenantId"] = df["groupTenantId"].fillna("").replace("", "(anonymous)")
    return df, data.get("params", {})

api_df,  api_params  = _load_grouped(GRP_DIR, "*sum_api_calls_by_tenant*")
inst_df, inst_params = _load_grouped(GRP_DIR, "*avg_instances_by_tenant*")

# Short label helper (reused from telemetry notebook)
def _short_label(tenant_id):
    if not tenant_id or tenant_id in ("(anonymous)",):
        return "(anonymous)"
    if len(tenant_id) <= 12:
        return tenant_id
    return tenant_id[:8] + "…"

if api_df is not None:
    # ── Query window: derived from params (what the API was asked for) ────────
    q_start_ms = api_params.get("startDate")
    q_end_ms   = api_params.get("endDate")
    if q_start_ms and q_end_ms:
        query_start = pd.Timestamp(q_start_ms, unit="ms", tz="UTC").date()
        query_end   = pd.Timestamp(q_end_ms,   unit="ms", tz="UTC").date()
        query_days  = (query_end - query_start).days
    else:
        # Fall back to data extent if params don't carry dates
        query_start = api_df["periodStart"].min().date()
        query_end   = api_df["periodEnd"].max().date()
        query_days  = (query_end - query_start).days

    # ── Active window: days that actually have non-zero data ──────────────────
    daily_totals = (
        api_df.groupby(api_df["periodStart"].dt.date)["periodQuantity"]
        .sum()
    )
    active_days_series = daily_totals[daily_totals > 0]
    active_days  = len(active_days_series)
    active_start = active_days_series.index.min() if active_days else None
    active_end   = active_days_series.index.max() if active_days else None
    gap_days     = query_days - active_days

    # ── Summary print ─────────────────────────────────────────────────────────
    n_tenants   = api_df["groupTenantId"].nunique()
    total_calls = api_df["periodQuantity"].sum()
    print(f"✓ api_calls  : {len(api_df)} rows  |  {n_tenants} tenants")
    print(f"  Query window : {query_start} → {query_end}  ({query_days} days)")
    if active_days and active_start:
        print(f"  Active period: {active_start} → {active_end}  ({active_days} active days, {gap_days} gap days)")
    print(f"  Total api_calls: {total_calls:,.0f}")
else:
    query_days = active_days = gap_days = 0
    active_start = active_end = query_start = query_end = None
    print("⚠ api_calls grouped data not found — run Fetch - Usage Data.ipynb first.")

if inst_df is not None:
    print(f"✓ instances  : {len(inst_df)} rows  |  {inst_df['groupTenantId'].nunique()} tenants")
else:
    print("⚠ instances grouped data not found — Model 2 (per-instance) will be skipped.")

# ── Per-tenant totals for billing calculations ────────────────────────────────
if api_df is not None:
    tenant_totals = (
        api_df.groupby("groupTenantId")["periodQuantity"]
        .sum()
        .reset_index()
        .rename(columns={"periodQuantity": "total_calls"})
        .sort_values("total_calls", ascending=False)
    )
    tenant_totals["label"] = tenant_totals["groupTenantId"].apply(_short_label)
    print(f"\nPer-tenant totals:")
    for _, row in tenant_totals.iterrows():
        print(f"  {row['label']:15s}  {row['total_calls']:>12,.0f} calls")

    # ── Data coverage summary ─────────────────────────────────────────────────
    if query_days > 0:
        pct_active = round(active_days / query_days * 100, 1)
        if gap_days > 0:
            print(f"  Coverage     : {active_days} active days ({pct_active}% of {query_days}-day query window)")
            print(f"ℹ  {gap_days} days in the query window had zero events — the service was not"
                  f" yet active or had no traffic. Billing is calculated over the {active_days} active days.")
        else:
            print(f"  Coverage     : {active_days} days — full query window covered ({pct_active}%)")

✓ api_calls  : 54 rows  |  4 tenants
  Query window : 2026-08-11 → 2026-08-25  (14 days)
  Active period: 2026-08-11 → 2026-08-24  (14 active days, 0 gap days)
  Total api_calls: 68,331,762
✓ instances  : 50 rows  |  4 tenants

Per-tenant totals:
  0fd2a329…          41,760,886 calls
  platform            9,091,133 calls
  7ef3d047…           9,068,822 calls
  ab475c00…           8,410,921 calls
  Coverage     : 14 days — full query window covered (100.0%)


## 4. Charts

### 4.1 Usage Baseline

### Computation

Sums `periodQuantity` (api_calls) per tenant across all active periods to produce a
grand total per tenant. Results are sorted ascending so the largest bar appears at the top.
Short labels (first 8 chars + …) are used on the Y axis; full tenant IDs are in hover.
This baseline is the input to all four billing models below.


In [4]:
fig_41 = None
if api_df is None:
    print('⚠ No api_calls data — run Fetch - Usage Data.ipynb first.')
else:
    COLOURS = ['#636EFA','#EF553B','#00CC96','#AB63FA','#FFA15A','#19D3F3']
    bar_data = tenant_totals.sort_values('total_calls', ascending=True)
    fig_41 = go.Figure(go.Bar(
        x=bar_data['total_calls'],
        y=bar_data['label'],
        orientation='h',
        text=[f'{v/1e6:.1f}M' for v in bar_data['total_calls']],
        textposition='outside',
        marker_color=COLOURS[:len(bar_data)],
        customdata=bar_data['groupTenantId'],
        hovertemplate='<b>%{customdata}</b><br>Total api_calls: %{x:,.0f}<extra></extra>',
    ))
    fig_41.update_layout(
        title=f'Billing period total — api_calls per tenant ({active_days}-day active window)',
        xaxis_title='Total api_calls', yaxis_title='Tenant',
        plot_bgcolor='white', showlegend=False,
        xaxis=dict(showgrid=True, gridcolor='#eee'),
        margin=dict(l=120, r=120, t=80, b=60),
        height=max(300, 60 + len(bar_data) * 55),
    )


### Chart Guide

**Purpose:** Establishes the usage facts before applying any pricing model.
Shows exactly how many API calls each tenant made during the active billing window.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Horizontal bar chart |
| **X axis** | Total `api_calls` across all active periods |
| **Y axis** | Tenant (shortened label) |
| **Bar label** | Value in millions (e.g. `27.5M`) |
| **Hover** | Full tenant UUID + exact call count |

**Insights:** The dominant consumer drives the most revenue under PayGo and the most overage
under contract models. A tenant with very few calls may be a candidate for a smaller tier.


In [5]:
if fig_41 is not None:
    fig_41.show()


### 4.2 Model 1 — Pay-as-you-go (PayGo)

### Computation

Multiplies each tenant's total `api_calls` by `PAYGO_RATE_PER_CALL` (set in Configuration).
`cost = total_calls × rate_per_call`. No minimum, no discount — cost scales linearly
with consumption. A platform revenue total is printed below the chart.


In [ ]:
fig_42 = None
if api_df is None:
    print("⚠ No api_calls data.")
else:
    billing = tenant_totals.copy()
    billing["cost_paygo"] = billing["total_calls"] * PAYGO_RATE_PER_CALL
    fig_42 = go.Figure(go.Bar(
        x=billing["label"],
        y=billing["cost_paygo"],
        text=[f"${v:,.2f}" for v in billing["cost_paygo"]],
        textposition="outside",
        marker_color=COLOURS[:len(billing)],
        customdata=list(zip(billing["groupTenantId"], billing["total_calls"])),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "api_calls: %{customdata[1]:,.0f}<br>"
            "PayGo cost: $%{y:,.2f}"
            "<extra></extra>"
        ),
    ))
    fig_42.update_layout(
        title=f"Model 1: PayGo — cost per tenant  (${PAYGO_RATE_PER_CALL*1e6:.2f} per 1M calls)",
        xaxis_title="Tenant", yaxis_title="Cost (USD)",
        yaxis_range=[0, billing["cost_paygo"].max() * 1.22],
        plot_bgcolor="white", showlegend=False,
        xaxis=dict(tickangle=-45),
        margin=dict(t=80, b=120, l=80, r=80),
    )
    total_paygo = billing["cost_paygo"].sum()
    print(f"Total platform revenue (PayGo): ${total_paygo:,.2f}")
    print(f"Rate applied: ${PAYGO_RATE_PER_CALL*1e6:.4f} per 1M calls")

### Chart Guide

**Purpose:** Shows what each tenant would pay if billed purely on consumption at a fixed rate.
The most transparent and proportional billing model.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Vertical bar chart |
| **X axis** | Tenant |
| **Y axis** | Cost in USD |
| **Bar label** | Exact dollar amount |
| **Hover** | Full tenant ID + call count + cost |

**Insights:** Bar heights are directly proportional to usage — the same shape as 4.1.
The dominant consumer pays proportionally more. Unpredictable for tenants with variable workloads.
> Rate: **$0.50 per 1M calls** (configurable in Section 2).


In [7]:
if fig_42 is not None:
    fig_42.show()


### 4.3 Model 2 — Per-instance Flat Rate

### Computation

Averages `periodQuantity` from the instances-by-tenant grouped dataset to get the
mean number of active instances per period per tenant. The average is rounded up with
`math.ceil` (partial instances bill as whole instances).
`cost = ceil(avg_instances) × INSTANCE_RATE`.


In [ ]:
fig_43 = None
if inst_df is None:
    print("⚠ No instances grouped data — Model 2 skipped.")
else:
    inst_totals = (
        inst_df.groupby("groupTenantId")["periodQuantity"]
        .mean()   # avg instances per period across the window
        .reset_index()
        .rename(columns={"periodQuantity": "avg_instances"})
    )
    inst_totals["avg_instances_rounded"] = inst_totals["avg_instances"].apply(math.ceil)
    inst_totals["cost_instance"] = inst_totals["avg_instances_rounded"] * INSTANCE_RATE
    inst_totals["label"] = inst_totals["groupTenantId"].apply(_short_label)
    fig_43 = go.Figure(go.Bar(
        x=inst_totals["label"],
        y=inst_totals["cost_instance"],
        text=[f"${v:,.2f}" for v in inst_totals["cost_instance"]],
        textposition="outside",
        marker_color=COLOURS[:len(inst_totals)],
        customdata=list(zip(inst_totals["groupTenantId"], inst_totals["avg_instances_rounded"])),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Avg instances: %{customdata[1]}<br>"
            "Instance cost: $%{y:,.2f}"
            "<extra></extra>"
        ),
    ))
    fig_43.update_layout(
        title=f"Model 2: Per-instance flat rate  (${INSTANCE_RATE:.0f}/instance/period)",
        xaxis_title="Tenant", yaxis_title="Cost (USD)",
        yaxis_range=[0, inst_totals["cost_instance"].max() * 1.22],
        plot_bgcolor="white", showlegend=False,
        xaxis=dict(tickangle=-45),
        margin=dict(t=80, b=120, l=80, r=80),
    )
    print(f"Total platform revenue (per-instance): ${inst_totals['cost_instance'].sum():,.2f}")

### Chart Guide

**Purpose:** Shows what each tenant would pay if charged a flat rate per running instance,
regardless of how much traffic those instances handle.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Vertical bar chart |
| **X axis** | Tenant |
| **Y axis** | Cost in USD |
| **Hover** | Full tenant ID + average instance count + cost |

**Insights:** A tenant with many low-traffic instances pays more than one with fewer
high-traffic instances. This model rewards efficient instance packing.
> Rate: **$200 per instance per billing period** (configurable in Section 2).


In [9]:
if fig_43 is not None:
    fig_43.show()


### 4.4 Model 3 — Contract / Pre-purchase

### Computation

Computes `pct_used = (total_calls / CONTRACT_CALLS) × 100` for each tenant.
Each tenant gets a gauge indicator showing what percentage of their committed volume
they consumed. The gauge arc turns red beyond 100% to flag over-commitment.
A threshold line at 100% marks the contract boundary.


In [10]:
fig_44 = None
if api_df is None:
    print("⚠ No api_calls data.")
else:
    billing = tenant_totals.copy()
    billing["cost_paygo"]    = billing["total_calls"] * PAYGO_RATE_PER_CALL
    billing["contract_cost"] = CONTRACT_CALLS * CONTRACT_RATE_PER_CALL
    billing["pct_used"]      = (billing["total_calls"] / CONTRACT_CALLS * 100).round(1)
    billing["paygo_equiv"]   = billing["total_calls"] * PAYGO_RATE_PER_CALL
    billing["saving"]        = billing["paygo_equiv"] - billing["contract_cost"]
    n = len(billing)
    ncols = min(2, n)
    nrows = math.ceil(n / ncols)
    # Wrap a tenant ID at every 20 chars so it fits inside a gauge cell
    def _wrap(tid, width=20):
        return "<br>".join(tid[i:i+width] for i in range(0, len(tid), width))
    # Gauges: % of contract consumed
    specs = [[{"type":"indicator"} for _ in range(ncols)] for _ in range(nrows)]
    fig_44 = make_subplots(rows=nrows, cols=ncols, specs=specs, vertical_spacing=0.30)
    for idx, (_, row) in enumerate(billing.iterrows()):
        r, c = divmod(idx, ncols)
        tid = _wrap(row["groupTenantId"])
        fig_44.add_trace(go.Indicator(
            mode="gauge+number+delta",
            value=row["pct_used"],
            title={
                "text": f"<b>{tid}</b><br><span style='font-size:.75em'>{row['total_calls']/1e6:.1f}M / {CONTRACT_CALLS/1e6:.0f}M calls</span>",
                "font": {"size": 12},
            },
            delta={"reference": 100, "valueformat": ".1f", "suffix": "%"},
            gauge={
                "axis":     {"range": [0, max(200, row["pct_used"]*1.2)]},
                "bar":      {"color": "#636EFA" if row["pct_used"] <= 100 else "#EF553B"},
                "steps":    [{"range":[0,100],"color":"#eef2ff"},{"range":[100,max(200,row["pct_used"]*1.2)],"color":"#fee2e2"}],
                "threshold":{"line":{"color":"red","width":3},"thickness":0.75,"value":100},
            },
            number={"suffix":"%","font":{"size":28}},
        ), row=r+1, col=c+1)
    fig_44.update_layout(
        height=400*nrows,
        title=dict(
            text=f"Model 3: Contract utilisation — {CONTRACT_CALLS/1e6:.0f}M call commitment per tenant",
            x=0.5, xanchor="center",
            pad=dict(b=30),
        ),
        paper_bgcolor="white", margin=dict(t=140,b=60,l=40,r=40),
    )

### Chart Guide

**Purpose:** Shows visually whether each tenant is under-utilising or over-running
their contracted volume — the key signal for contract renewal conversations.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Gauge indicators (one per tenant) |
| **Needle** | Percentage of contract consumed |
| **Blue arc (0–100%)** | Within contract |
| **Red arc (>100%)** | Over contract |
| **Delta** | Difference from 100% contract target |

**Insights:** Below 100% means the tenant pre-paid for more than they used — they overpaid.
Above 100% means they exceeded their commitment — overage charges apply in Model 4.
> Contract: **20M calls at $8/1M** (configurable in Section 2).


In [11]:
if fig_44 is not None:
    fig_44.show()


### 4.5 Model 4 — Contract + Overage

### Computation

The contract is a **fixed commitment**: every tenant pays `CONTRACT_CALLS × CONTRACT_RATE_PER_CALL`
regardless of whether they reach the threshold.
Calls beyond `CONTRACT_CALLS` are billed additionally at the overage rate:

- `cost_contract = CONTRACT_CALLS × CONTRACT_RATE_PER_CALL` — flat fee for every tenant
- `overage_calls = max(0, total_calls − CONTRACT_CALLS)` — calls above the threshold
- `cost_overage  = overage_calls × OVERAGE_RATE_PER_CALL`
- `cost_total    = cost_contract + cost_overage`

The two cost components are stacked to show the total bill with the split clearly visible.
A total cost annotation is placed above each bar.


In [12]:
fig_45 = None
if api_df is None:
    print("⚠ No api_calls data.")
else:
    billing = tenant_totals.copy()
    billing["overage_calls"] = (billing["total_calls"] - CONTRACT_CALLS).clip(lower=0)
    billing["cost_contract"] = CONTRACT_CALLS * CONTRACT_RATE_PER_CALL  # flat commitment — same for every tenant
    billing["cost_overage"]  = billing["overage_calls"] * OVERAGE_RATE_PER_CALL
    billing["cost_total"]    = billing["cost_contract"] + billing["cost_overage"]
    fig_45 = go.Figure()
    fig_45.add_trace(go.Bar(
        name="In contract",
        x=billing["label"],
        y=billing["cost_contract"],
        marker_color="#636EFA",
        customdata=list(zip(billing["groupTenantId"], [CONTRACT_CALLS] * len(billing))),
        hovertemplate="<b>%{customdata[0]}</b><br>Contract commitment: %{customdata[1]:,.0f} calls<br>Fixed cost: $%{y:,.2f}<extra></extra>",
    ))
    fig_45.add_trace(go.Bar(
        name="Overage",
        x=billing["label"],
        y=billing["cost_overage"],
        marker_color="#EF553B",
        customdata=list(zip(billing["groupTenantId"], billing["overage_calls"])),
        hovertemplate="<b>%{customdata[0]}</b><br>Overage calls: %{customdata[1]:,.0f}<br>Overage cost: $%{y:,.2f}<extra></extra>",
    ))
    # Total cost label on top of each stacked bar
    for _, row in billing.iterrows():
        fig_45.add_annotation(
            x=row["label"], y=row["cost_total"],
            text=f"${row['cost_total']:,.2f}",
            showarrow=False, yshift=10,
            font=dict(size=11, color="#333"),
        )
    fig_45.update_layout(
        barmode="stack",
        title=(
            f"Model 4: Contract + Overage  "
            f"(contract: ${CONTRACT_RATE_PER_CALL*1e6:.2f}/1M · overage: ${OVERAGE_RATE_PER_CALL*1e6:.2f}/1M)"
        ),
        xaxis_title="Tenant", yaxis_title="Cost (USD)",
        yaxis_range=[0, billing["cost_total"].max() * 1.25],
        plot_bgcolor="white",
        legend=dict(orientation="h",y=-0.18,x=0.5,xanchor="center"),
        margin=dict(t=80, b=100, l=80, r=80),
    )
    print("\nContract + Overage breakdown:")
    for _, row in billing.iterrows():
        print(f"  {row['label']:15s}  total={row['total_calls']/1e6:6.1f}M  "
              f"contract=${row['cost_contract']:7.2f}  overage=${row['cost_overage']:7.2f}  "
              f"total=${row['cost_total']:7.2f}")


Contract + Overage breakdown:
  0fd2a329…        total=  41.8M  contract=$ 160.00  overage=$  15.23  total=$ 175.23
  platform         total=   9.1M  contract=$ 160.00  overage=$   0.00  total=$ 160.00
  7ef3d047…        total=   9.1M  contract=$ 160.00  overage=$   0.00  total=$ 160.00
  ab475c00…        total=   8.4M  contract=$ 160.00  overage=$   0.00  total=$ 160.00


### Chart Guide

**Purpose:** Shows the cost breakdown between the contracted portion and the overage,
making it easy to see which tenants are significantly over-running their commitments.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Stacked bar chart |
| **X axis** | Tenant |
| **Y axis** | Cost in USD |
| **Blue segment** | In-contract portion (at contract rate) |
| **Red segment** | Overage portion (at overage rate) |
| **Top label** | Total cost |

**Insights:** A large red segment means a tenant significantly exceeded their commitment —
a signal to move them to a higher contract tier.
A tenant with no red segment is within contract and may be a candidate for a smaller tier.
> Contract: $8/1M · Overage: $0.70/1M (configurable in Section 2).


In [13]:
if fig_45 is not None:
    fig_45.show()


### 4.6 Model Comparison

### Computation

Computes all four model costs for each tenant in one DataFrame, then melts to long format
for a grouped bar chart. Per-instance costs are joined from the instances dataset if available.
A summary table is printed below showing exact dollar amounts including a TOTAL row.


In [14]:
fig_46 = None
if api_df is None:
    print("⚠ No api_calls data.")
else:
    comp = tenant_totals.copy()
    comp["PayGo"]               = comp["total_calls"] * PAYGO_RATE_PER_CALL
    comp["Contract"]            = CONTRACT_CALLS * CONTRACT_RATE_PER_CALL
    comp["Contract + Overage"]  = (
        CONTRACT_CALLS * CONTRACT_RATE_PER_CALL  # flat commitment
        + (comp["total_calls"] - CONTRACT_CALLS).clip(lower=0) * OVERAGE_RATE_PER_CALL
    )
    if inst_df is not None:
        inst_map = (
            inst_df.groupby("groupTenantId")["periodQuantity"]
            .mean().apply(math.ceil)
            .to_dict()
        )
        comp["Per-instance"] = comp["groupTenantId"].map(inst_map).fillna(1) * INSTANCE_RATE
    else:
        comp["Per-instance"] = float("nan")
    models = ["PayGo", "Per-instance", "Contract", "Contract + Overage"]
    models = [m for m in models if m in comp.columns and not comp[m].isna().all()]
    # Grouped bar
    long = comp.melt(id_vars=["label","groupTenantId"], value_vars=models, var_name="Model", value_name="Cost")
    MCOLOURS = {"PayGo":"#636EFA","Per-instance":"#00CC96","Contract":"#FFA15A","Contract + Overage":"#EF553B"}
    fig_46 = px.bar(
        long, x="label", y="Cost", color="Model", barmode="group",
        text=long["Cost"].apply(lambda v: f"${v:,.1f}"),
        title="Model comparison — cost per tenant under each pricing model",
        labels={"label":"Tenant","Cost":"Cost (USD)"},
        color_discrete_map=MCOLOURS,
        custom_data=["groupTenantId"],
    )
    fig_46.update_traces(textposition="outside", cliponaxis=False,
        hovertemplate="<b>%{customdata[0]}</b><br>%{x}<br>%{fullData.name}: $%{y:,.2f}<extra></extra>")
    fig_46.update_layout(
        yaxis_range=[0, long["Cost"].max() * 1.28],
        plot_bgcolor="white",
        legend=dict(orientation="h",y=-0.20,x=0.5,xanchor="center",title="Model"),
        margin=dict(t=80, b=120, l=80, r=80),
    )
    # Summary table
    summary = comp[["label"] + models].copy()
    summary.columns = ["Tenant"] + models
    summary.loc[len(summary)] = ["TOTAL"] + [summary[m].sum() for m in models]
    for m in models:
        summary[m] = summary[m].apply(lambda v: f"${v:,.2f}")
    print("\nCost comparison summary:")
    display(summary.set_index("Tenant"))


Cost comparison summary:


,PayGo,Per-instance,Contract,Contract + Overage
Tenant,,,,
0fd2a329…,$20.88,$200.00,$160.00,$175.23
platform,$4.55,$200.00,$160.00,$160.00
7ef3d047…,$4.53,$200.00,$160.00,$160.00
ab475c00…,$4.21,$200.00,$160.00,$160.00
TOTAL,$34.17,$800.00,$640.00,$655.23


### Chart Guide

**Purpose:** The most useful view for an MSP sales or commercial conversation — shows
what each tenant would pay under every model side by side.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Grouped bar chart |
| **X axis** | Tenant |
| **Y axis** | Cost in USD |
| **Colour** | One colour per pricing model |
| **Hover** | Full tenant ID + model name + exact cost |

**Insights:** A tenant where **PayGo < Contract** might prefer PayGo or a smaller contract.
A tenant where **Contract + Overage > PayGo** significantly exceeded their commitment —
time to move them up a tier. The totals row shows which model generates the most platform revenue.
